## Parameters

Configure the following parameters before running the notebook:

In [ ]:
import json
import logging
import pandas as pd
import numpy as np
import pickle
import re
import time
import os
import uuid
import getpass
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Optional, Any, Tuple
import pubchempy as pcp
from tqdm.notebook import tqdm
import sys

In [ ]:
# Required: Input TSV file with experimental data
tsv_file_path = "/global/homes/b/bkieft/metatlas-data/HILIC/HILIC_QCv7_positive.tsv"

# Required: Output directory for JSON and cache files
output_dir = "/global/homes/b/bkieft/metatlas2/data/databases/compounds"
output_path = Path(output_dir)

# Required: Backup existing database files before modifying
backup_needed = False

# Optional: Cache settings
use_cache = True  # Use cached PubChem data if available
tsv_name = os.path.basename(tsv_file_path).replace(".tsv", "")
cache_filename = f"{tsv_name}_pubchem_cache.pkl"  # Prefix for cache file

# Optional: Name filtering (following your example patterns)
missing_name_default = "Undefined"
problematic_prefixes = (
    "nan", "Untitled", "Oprea", "Opera", "AKO", "CHEMBL", "SR-", "SCHEM", 
    "EU-", "MLS", "NSC", "ChemDiv", "ST0", "TimTec", "HMS", "BIM", "CB", 
    "CCG-", "Cambridge", "SMR", "AB0", "BRD-", "NCG", "BDBM", "CBKinase", 
    "BAS ", "ZINC", "GNF", "SQX", "CDS", "STK", "NCI", "TNP", "Boc-Tyr-OH", 
    "PD", "UNM", "BSP", "CCRIS", "MFCD", "IDI", "ST5", "AC1", "WAY-", "KUC", 
    "DTXSID", "MixCom", "CK-", "ASN ", "MMV", "SKI-", "VU", "SMSF", "Bio2", 
    "REGID", "SDCC", "BCBc", "SMP", "TCMDC", "cid_", "BCP", "AST ", "SY0", 
    "AM-", "IFLab", "Cream"
)

# Optional: PubChem API settings
max_compounds_per_request = 5  # Limit compounds per PubChem request
request_delay = 0.2  # Delay between requests (seconds) to be respectful

# Creator information
creator_name = getpass.getuser()
creation_notes = "Core compounds database initialization"
timestamp = datetime.now().isoformat()

print("Parameters configured!")
print(f"Input: {tsv_file_path}")
print(f"Output: {output_dir}")
print(f"Current user: {creator_name}")
print(f"Timestamp: {timestamp}")

## Load and Validate TSV Data

In [ ]:
# Load and validate TSV data
print(f"Loading TSV data from: {tsv_file_path}")
delimiter = '\t' if tsv_file_path.endswith(('.tsv', '.tab', '.txt')) else ','
compounds_df = pd.read_csv(tsv_file_path, sep=delimiter)
print(f"Loaded {len(compounds_df)} rows from TSV")

# Check for required columns (only core compound identification needed)
required_columns = ['inchi_key', 'label']
missing_columns = [col for col in required_columns if col not in compounds_df.columns]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

# Get unique InChI keys for PubChem lookup
unique_inchi_keys = compounds_df['inchi_key'].dropna().unique()
print(f"\nUnique InChI keys to process: {len(unique_inchi_keys)}")

## PubChem Data Retrieval Functions

In [ ]:
def get_pubchem_data(inchi_key: str) -> Dict[str, Any]:
    """
    Get comprehensive compound data from PubChem using InChI key.
    """
    try:
        # Get CID from InChI key
        cid_result = pcp.get_compounds(inchi_key, namespace='inchikey', 
                                        as_dataframe=True, listkey_count=max_compounds_per_request)

        if cid_result.empty:
            return None
            
        cid_result = cid_result.reset_index()
        cid = cid_result['cid'].to_string(index=False)
        
        # Handle multiple CIDs
        if "\n" in cid:
            cid = (cid.rstrip().split('\n'))[-1]
        
        # Extract SMILES if available due to broken API response
        try:
            cid_result_subset = cid_result[cid_result['cid'] == int(cid)]
            cid_result_subset_dict = cid_result_subset.to_dict()
            if "record" in cid_result_subset_dict:
                if "props" in cid_result_subset_dict["record"][0]:
                    cid_list = [i for i in cid_result_subset_dict["record"][0]["props"] if i["urn"]["label"] == "SMILES"]
                    if cid_list:
                        smiles = str(cid_list[0]['value']['sval'])
        except:
            smiles = ""

        # Get detailed compound information
        compound = pcp.Compound.from_cid(cid)
        
        # Extract all available properties using direct attribute access
        compound_data = {
            "pubchem_cid": str(compound.cid) if compound.cid else "",
            "iupac_name": compound.iupac_name or "",
            "synonyms": compound.synonyms or [],
            "inchi": compound.inchi or "",
            "inchi_key": compound.inchikey or inchi_key,
            "smiles": smiles if smiles else "",
            "formula": compound.molecular_formula or "",
            "mono_isotopic_molecular_weight": str(compound.monoisotopic_mass) if compound.monoisotopic_mass else "",
            "molecular_weight": str(compound.molecular_weight) if compound.molecular_weight else "",
            
            # Database identifiers - extract from synonyms
            "cas_number": "",
            "chebi_id": "",
            "hmdb_id": "",
            "kegg_id": "",
            "lipidmaps_id": "",
            "metacyc_id": "",
            
            # Metadata
            "pubchem_retrieval_date": timestamp,
            "pubchem_compound_url": f"https://pubchem.ncbi.nlm.nih.gov/compound/{compound.cid}" if compound.cid else ""
        }
        
        # Extract database IDs from synonyms
        if compound.synonyms:
            for synonym in compound.synonyms:
                syn_upper = synonym.upper()
                if not compound_data["cas_number"] and '-' in synonym and len(synonym.split('-')) == 3:
                    parts = synonym.split('-')
                    if all(part.isdigit() for part in parts):
                        compound_data["cas_number"] = synonym
                elif not compound_data["chebi_id"] and 'CHEBI:' in syn_upper:
                    compound_data["chebi_id"] = synonym
                elif not compound_data["hmdb_id"] and 'HMDB' in syn_upper:
                    compound_data["hmdb_id"] = synonym
                elif not compound_data["kegg_id"] and len(synonym) == 6 and synonym.startswith('C') and synonym[1:].isdigit():
                    compound_data["kegg_id"] = synonym
        
        return compound_data
        
    except Exception as e:
        print(f"Error retrieving PubChem data for {inchi_key}: {e}")
        return None

def filter_synonym_list(synonyms: List[str]) -> str:
    """
    Filter synonym list to find the best name, based on user's existing logic.
    """
    if not synonyms or synonyms == [missing_name_default]:
        return missing_name_default
    
    if isinstance(synonyms, str):
        synonyms = [synonyms]
    
    # Remove problematic prefixes
    filtered_synonyms = [x for x in synonyms if not x.startswith(problematic_prefixes)]
    filtered_synonyms = [x for x in filtered_synonyms if not "Cream" in x]
    
    # Remove entries that are mostly digits
    filtered_synonyms = [x for x in filtered_synonyms 
                        if not x.replace("-", "").replace(re.compile('^A-Z').pattern, "").isdigit()]
    
    # Remove single letter followed by digits
    filtered_synonyms = [x for x in filtered_synonyms 
                        if not re.sub(r'\b[A-Z]{1}',"",x).isdigit()]
    
    # Remove long alphanumeric codes (like InChI keys in names)
    filtered_synonyms = [x for x in filtered_synonyms 
                        if not re.compile(r'\b[A-Z]{14}\b-\b[A-Z]{10}\b-\b[A-Z]{1}\b').search(x)]
    
    # Remove F-codes
    filtered_synonyms = [x for x in filtered_synonyms 
                        if not re.compile(r'^F\d{4}-\d{4}').search(x)]
    
    if not filtered_synonyms:
        return missing_name_default
    
    # Return the shortest remaining synonym (likely to be the best)
    return min(filtered_synonyms, key=len)

def load_or_create_pubchem_cache() -> Dict[str, Dict]:
    """Load existing PubChem cache or create new one"""
    cache_file = output_path / "pubchem_cache" / cache_filename
    
    if use_cache and cache_file.exists():
        try:
            with open(cache_file, 'rb') as f:
                cache = pickle.load(f)
            print(f"Loaded PubChem cache with {len(cache)} entries from {cache_file}")
            return cache
        except Exception as e:
            print(f"Error loading cache: {e}. Creating new cache.")
    
    print("Creating new PubChem cache")
    return {}

def save_pubchem_cache(cache: Dict[str, Dict]) -> None:
    """Save PubChem cache to file"""
    cache_file = output_path / "pubchem_cache" / cache_filename
    
    try:
        with open(cache_file, 'wb') as f:
            pickle.dump(cache, f)
        print(f"Saved PubChem cache with {len(cache)} entries to {cache_file}")
    except Exception as e:
        print(f"Error saving cache: {e}")

# New functions for normalized database structure
def load_normalized_databases() -> Tuple[Dict, Dict, Dict, Dict, Dict]:
    """
    Load all normalized database files and indexes.
    
    Returns:
        Tuple of (compounds, methods, rt_references, mz_references, indexes)
    """
    compounds_file = output_path / "Compounds.json"
    methods_file = output_path / "Methods.json"
    rt_references_file = output_path / "RT_References.json"
    mz_references_file = output_path / "MZ_References.json"
    indexes_file = output_path / "Indexes.json"
    
    # Load or initialize each database
    compounds = {}
    methods = {}
    rt_references = {}
    mz_references = {}
    indexes = {
        "compound_by_inchi_key": {},
        "rt_references_by_compound": {},
        "mz_references_by_compound": {},
        "references_by_method": {}
    }
    
    # Load compounds
    if compounds_file.exists():
        try:
            with open(compounds_file, 'r') as f:
                compounds = json.load(f)
            print(f"Loaded {len(compounds)} compounds")
        except Exception as e:
            print(f"Error loading compounds: {e}")
    
    # Load methods
    if methods_file.exists():
        try:
            with open(methods_file, 'r') as f:
                methods = json.load(f)
            print(f"Loaded {len(methods)} methods")
        except Exception as e:
            print(f"Error loading methods: {e}")
    
    # Load RT references
    if rt_references_file.exists():
        try:
            with open(rt_references_file, 'r') as f:
                rt_references = json.load(f)
            print(f"Loaded {len(rt_references)} RT references")
        except Exception as e:
            print(f"Error loading RT references: {e}")
    
    # Load MZ references
    if mz_references_file.exists():
        try:
            with open(mz_references_file, 'r') as f:
                mz_references = json.load(f)
            print(f"Loaded {len(mz_references)} MZ references")
        except Exception as e:
            print(f"Error loading MZ references: {e}")
    
    # Load indexes
    if indexes_file.exists():
        try:
            with open(indexes_file, 'r') as f:
                indexes = json.load(f)
            print(f"Loaded indexes")
        except Exception as e:
            print(f"Error loading indexes: {e}")
    
    return compounds, methods, rt_references, mz_references, indexes

def save_normalized_databases(compounds: Dict, methods: Dict, rt_references: Dict, 
                            mz_references: Dict, indexes: Dict) -> None:
    """Save all normalized database files"""
    
    # Save compounds
    compounds_file = output_path / "Compounds.json"
    with open(compounds_file, 'w') as f:
        json.dump(compounds, f, indent=2)
    
    # Save methods
    methods_file = output_path / "Methods.json"
    with open(methods_file, 'w') as f:
        json.dump(methods, f, indent=2)
    
    # Save RT references
    rt_references_file = output_path / "RT_References.json"
    with open(rt_references_file, 'w') as f:
        json.dump(rt_references, f, indent=2)
    
    # Save MZ references
    mz_references_file = output_path / "MZ_References.json"
    with open(mz_references_file, 'w') as f:
        json.dump(mz_references, f, indent=2)
    
    # Save indexes
    indexes_file = output_path / "Indexes.json"
    with open(indexes_file, 'w') as f:
        json.dump(indexes, f, indent=2)
    
    print(f"✓ All normalized database files saved to {output_path}")

def find_or_create_method(methods: Dict, indexes: Dict, method_config: Dict) -> str:
    """
    Find existing method or create new one based on method configuration.
    
    Returns:
        method_uid of the found or created method
    """
    # Look for existing method with same chromatography and polarity
    method_key = f"{method_config['chromatography']}_{method_config['polarity']}"
    
    # Check if method already exists
    for method_uid, method_data in methods.items():
        if (method_data.get('chromatography') == method_config['chromatography'] and
            method_data.get('polarity') == method_config['polarity']):
            print(f"Found existing method: {method_uid} ({method_data.get('method_name')})")
            return method_uid
    
    # Create new method
    method_uid = f"method-{uuid.uuid4().hex[:16]}"
    methods[method_uid] = {
        "method_uid": method_uid,
        "method_name": method_config.get('method_name', f"{method_config['chromatography']}_{method_config['polarity']}"),
        "chromatography": method_config['chromatography'],
        "polarity": method_config['polarity'],
        "column": method_config.get('column', ''),
        "mobile_phase": method_config.get('mobile_phase', ''),
        "instrument": method_config.get('instrument', ''),
        "method_notes": method_config.get('method_notes', ''),
        "created_by": creator_name,
        "creation_time": timestamp,
        "last_modified": timestamp
    }
    
    # Initialize method in indexes
    indexes["references_by_method"][method_uid] = {
        "rt_references": [],
        "mz_references": []
    }
    
    print(f"Created new method: {method_uid} ({method_config['method_name']})")
    return method_uid

def update_indexes(indexes: Dict, compound_uid: str, inchi_key: str, 
                  rt_uid: str = None, mz_uid: str = None, method_uid: str = None) -> None:
    """Update all indexes with new references"""
    
    # Update compound by InChI key index
    if inchi_key not in indexes["compound_by_inchi_key"]:
        indexes["compound_by_inchi_key"][inchi_key] = []
    if compound_uid not in indexes["compound_by_inchi_key"][inchi_key]:
        indexes["compound_by_inchi_key"][inchi_key].append(compound_uid)
    
    # Update RT references by compound
    if compound_uid not in indexes["rt_references_by_compound"]:
        indexes["rt_references_by_compound"][compound_uid] = []
    if rt_uid and rt_uid not in indexes["rt_references_by_compound"][compound_uid]:
        indexes["rt_references_by_compound"][compound_uid].append(rt_uid)
    
    # Update MZ references by compound
    if compound_uid not in indexes["mz_references_by_compound"]:
        indexes["mz_references_by_compound"][compound_uid] = []
    if mz_uid and mz_uid not in indexes["mz_references_by_compound"][compound_uid]:
        indexes["mz_references_by_compound"][compound_uid].append(mz_uid)
    
    # Update references by method
    if method_uid and method_uid not in indexes["references_by_method"]:
        indexes["references_by_method"][method_uid] = {"rt_references": [], "mz_references": []}
    
    if method_uid and rt_uid and rt_uid not in indexes["references_by_method"][method_uid]["rt_references"]:
        indexes["references_by_method"][method_uid]["rt_references"].append(rt_uid)
    
    if method_uid and mz_uid and mz_uid not in indexes["references_by_method"][method_uid]["mz_references"]:
        indexes["references_by_method"][method_uid]["mz_references"].append(mz_uid)


# Functions for core compounds database
def load_compounds_database() -> Tuple[Dict, Dict]:
    """
    Load existing compounds database and index.
    
    Returns:
        Tuple of (compounds, compound_index)
    """
    compounds_file = output_path / "Compounds.json"
    compound_index_file = output_path / "Compounds_Index.json"
    
    compounds = {}
    compound_index = {"compound_by_inchi_key": {}}
    
    # Load compounds
    if compounds_file.exists():
        try:
            with open(compounds_file, 'r') as f:
                compounds = json.load(f)
            print(f"Loaded {len(compounds)} existing compounds")
        except Exception as e:
            print(f"Error loading compounds: {e}")
    
    # Load or rebuild index
    if compound_index_file.exists():
        try:
            with open(compound_index_file, 'r') as f:
                compound_index = json.load(f)
            print(f"Loaded compound index")
        except Exception as e:
            print(f"Error loading compound index: {e}")
            compound_index = {"compound_by_inchi_key": {}}
    
    # Rebuild index if empty or corrupted
    if not compound_index.get("compound_by_inchi_key"):
        print("Rebuilding compound index...")
        compound_index = {"compound_by_inchi_key": {}}
        for compound_uid, compound_data in compounds.items():
            inchi_key = compound_data.get('inchi_key')
            if inchi_key:
                if inchi_key not in compound_index["compound_by_inchi_key"]:
                    compound_index["compound_by_inchi_key"][inchi_key] = []
                if compound_uid not in compound_index["compound_by_inchi_key"][inchi_key]:
                    compound_index["compound_by_inchi_key"][inchi_key].append(compound_uid)
    
    return compounds, compound_index

def save_compounds_database(compounds: Dict, compound_index: Dict) -> None:
    """Save compounds database and index"""
    
    # Save compounds
    compounds_file = output_path / "Compounds.json"
    with open(compounds_file, 'w') as f:
        json.dump(compounds, f, indent=2)
    
    # Save compound index
    compound_index_file = output_path / "Compounds_Index.json"
    with open(compound_index_file, 'w') as f:
        json.dump(compound_index, f, indent=2)
    
    print(f"✓ Compounds database saved to {output_path}")
    print(f"  - {len(compounds)} compounds in Compounds.json")
    print(f"  - {len(compound_index['compound_by_inchi_key'])} InChI keys indexed")

def update_compound_index(compound_index: Dict, compound_uid: str, inchi_key: str) -> None:
    """Update compound index with new compound"""
    if inchi_key not in compound_index["compound_by_inchi_key"]:
        compound_index["compound_by_inchi_key"][inchi_key] = []
    if compound_uid not in compound_index["compound_by_inchi_key"][inchi_key]:
        compound_index["compound_by_inchi_key"][inchi_key].append(compound_uid)

def find_existing_compound(compounds: Dict, compound_index: Dict, inchi_key: str) -> Optional[str]:
    """
    Find existing compound by InChI key.
    
    Returns:
        compound_uid of existing compound, or None if not found
    """
    existing_uids = compound_index["compound_by_inchi_key"].get(inchi_key, [])
    
    if not existing_uids:
        return None
    
    # If multiple compounds exist for this InChI key, return the most recent one
    if len(existing_uids) == 1:
        return existing_uids[0]
    
    # Return most recently modified compound
    most_recent_uid = max(existing_uids, 
                         key=lambda uid: compounds[uid].get('last_modified', ''))
    
    print(f"Info: Found {len(existing_uids)} compounds for InChI key {inchi_key[:20]}..., using most recent: {most_recent_uid}")
    return most_recent_uid

## Retrieve PubChem Data for All Compounds

In [ ]:
# Load existing cache
pubchem_cache = load_or_create_pubchem_cache()

# Check which compounds need PubChem lookup
compounds_to_fetch = [key for key in unique_inchi_keys if key not in pubchem_cache]
compounds_in_cache = [key for key in unique_inchi_keys if key in pubchem_cache]

print(f"Compounds already in cache: {len(compounds_in_cache)}")
print(f"Compounds needing PubChem lookup: {len(compounds_to_fetch)}")

if compounds_to_fetch:
    print(f"\nFetching PubChem data for {len(compounds_to_fetch)} compounds...")
    print("This may take several minutes depending on the number of compounds.")
    
    # Fetch data for compounds not in cache
    for inchi_key in tqdm(compounds_to_fetch, desc="Fetching PubChem data"):
        
        if inchi_key in pubchem_cache:
            continue  # Skip if already processed
        
        # Get PubChem data
        pubchem_data = get_pubchem_data(inchi_key)
        
        if pubchem_data:
            pubchem_cache[inchi_key] = pubchem_data
            #print(f"✓ Retrieved data for {inchi_key}")
        else:
            print(f"No PubChem data found for {inchi_key}")
            # Store empty entry to avoid re-querying
            pubchem_cache[inchi_key] = {
                "inchi_key": inchi_key,
                "pubchem_cid": "",
                "iupac_name": "",
                "synonyms": [missing_name_default],
                "error": "not_found_in_pubchem"
            }
        
        # Be respectful to PubChem API
        time.sleep(request_delay)
    
    # Save updated cache
    save_pubchem_cache(pubchem_cache)
    
else:
    print("All compounds already in cache!")

print(f"\nPubChem data retrieval complete!")
print(f"Total compounds in cache: {len(pubchem_cache)}")

# Show examples
successful_retrievals = [k for k, v in pubchem_cache.items() 
                        if v.get('pubchem_cid') and v.get('pubchem_cid') != '']
failed_retrievals = [k for k, v in pubchem_cache.items() 
                    if not v.get('pubchem_cid') or v.get('pubchem_cid') == '']

print(f"Successful PubChem retrievals: {len(successful_retrievals)}")
print(f"Failed retrievals: {len(failed_retrievals)}")

if failed_retrievals:
    print(f"\nSome compounds not found in PubChem: {failed_retrievals[:5]}...")
    print("These will be created with minimal information from the TSV.")

## Create Compounds Database JSON Structure

## Load Existing Database (Optional)

Load existing compounds database and index if they exist, otherwise start fresh:

In [ ]:
# Load existing compounds database
compounds, compound_index = load_compounds_database()

print(f"\nStarting Database State:")
print(f"   Existing compounds: {len(compounds)}")
print(f"   InChI keys indexed: {len(compound_index['compound_by_inchi_key'])}")

# Check which InChI keys from table are already in database
existing_inchi_keys = set(compound_index['compound_by_inchi_key'].keys())
new_inchi_keys = set(unique_inchi_keys) - existing_inchi_keys
overlapping_inchi_keys = set(unique_inchi_keys) & existing_inchi_keys

print(f"   InChI keys from table already in database: {len(overlapping_inchi_keys)}")
print(f"   New InChI keys from table to be added: {len(new_inchi_keys)}")

In [ ]:
def create_compound_entry(inchi_key: str, compounds_df: pd.Series, pubchem_data: Dict) -> Dict[str, Any]:
    """
    Create a core compound entry with only chemical properties.
    """
    
    # Generate unique UID for this compound entry
    compound_uid = f"comp-{uuid.uuid4().hex[:16]}"
    
    # Create the compound entry dictionary with core chemical data only
    compound_entry = {
        "compound_uid": compound_uid,
        "name": str(compounds_df.get('label', missing_name_default)),
        "inchi_key": inchi_key,
        "inchi": pubchem_data.get('inchi', ''),
        "smiles": pubchem_data.get('smiles', ''),
        "formula": pubchem_data.get('formula', ''),
        "mono_isotopic_molecular_weight": pubchem_data.get('mono_isotopic_molecular_weight', ''),
        "molecular_weight": pubchem_data.get('molecular_weight', ''),
        "iupac_name": pubchem_data.get('iupac_name', ''),
        "pubchem_name": filter_synonym_list(pubchem_data.get('synonyms', [])),
        "synonyms": "; ".join(pubchem_data.get('synonyms', [])) if pubchem_data.get('synonyms') else '',
        
        # Database identifiers
        "pubchem_cid": pubchem_data.get('pubchem_cid', ''),
        "cas_number": pubchem_data.get('cas_number', ''),
        "chebi_id": pubchem_data.get('chebi_id', ''),
        "hmdb_id": pubchem_data.get('hmdb_id', ''),
        "kegg_id": pubchem_data.get('kegg_id', ''),
        "lipidmaps_id": pubchem_data.get('lipidmaps_id', ''),
        "metacyc_id": pubchem_data.get('metacyc_id', ''),
        
        # Metadata
        "data_source": "",
        "pubchem_url": pubchem_data.get('pubchem_compound_url', ''),
        "pubchem_retrieval_date": pubchem_data.get('pubchem_retrieval_date', ''),
        "created_by": creator_name,
        "creation_time": timestamp,
        "last_modified": timestamp,
        "notes": creation_notes
    }
    
    return compound_entry

In [ ]:
if len(new_inchi_keys) == 0:
    print("No new compounds to add. Database is up to date!")
    sys.exit(1)

# Process all compounds to create core database
print("Processing compounds to create core database...")

processing_stats = {
    "total_rows_processed": 0,
    "new_compounds_created": 0,
    "existing_compounds_found": 0
}

# Process each row in the TSV
for idx, row in tqdm(compounds_df.iterrows(), total=len(compounds_df), desc="Processing compounds"):
    inchi_key = row['inchi_key']
    
    if pd.isna(inchi_key) or inchi_key == '':
        print(f"Warning: Skipping row {idx} - missing InChI key")
        continue
    
    processing_stats["total_rows_processed"] += 1
    
    # Get PubChem data for this compound
    pubchem_data = pubchem_cache.get(inchi_key, {})
    
    # Check if compound already exists
    existing_compound_uid = find_existing_compound(compounds, compound_index, inchi_key)
    
    if existing_compound_uid:
        # Use existing compound
        compound_uid = existing_compound_uid
        processing_stats["existing_compounds_found"] += 1
        
        # Update last_modified time for the existing compound
        compounds[compound_uid]["last_modified"] = timestamp
        compounds[compound_uid]["last_modified_by"] = creator_name
        
    else:
        # Create new compound
        new_compound = create_compound_entry(inchi_key, row, pubchem_data)
        compound_uid = new_compound["compound_uid"]
        
        # Store the new compound
        compounds[compound_uid] = new_compound
        processing_stats["new_compounds_created"] += 1
        
        # Update compound index
        update_compound_index(compound_index, compound_uid, inchi_key)

print("✓ Compound processing complete!")
print(f"\nProcessing Statistics:")
for key, value in processing_stats.items():
    print(f"  {key.replace('_', ' ').title()}: {value}")

print(f"\nFinal Database Summary:")
print(f"   Total compounds: {len(compounds)}")
print(f"   Unique InChI keys: {len(compound_index['compound_by_inchi_key'])}")

# Data integrity check
print(f"\nData Integrity Check:")
compounds_with_valid_inchi = sum(1 for c in compounds.values() 
                                if c.get('inchi_key') and len(c.get('inchi_key', '')) == 27)
print(f"   Compounds with valid InChI keys: {compounds_with_valid_inchi}/{len(compounds)}")

compounds_with_names = sum(1 for c in compounds.values() 
                          if c.get('name') and c['name'] != missing_name_default)
print(f"   Compounds with proper names: {compounds_with_names}/{len(compounds)}")

In [ ]:
# Create backups and save compounds database
timestamp_str = datetime.now().strftime("%Y%m%d%H%M%S")

# Initialize backup tracking
backup_created = False
backup_dir = output_path / f"backup_{timestamp_str}"

# Check if compounds database exists to backup
compounds_file = output_path / "Compounds.json"
compound_index_file = output_path / "Compounds_Index.json"
metadata_file = output_path / "Compounds_Metadata.json"

existing_files = []
if compounds_file.exists():
    existing_files.append("Compounds.json")
if compound_index_file.exists():
    existing_files.append("Compounds_Index.json")
if metadata_file.exists():
    existing_files.append("Compounds_Metadata.json")

# Create backups only if requested and files exist
if backup_needed and existing_files:
    backup_dir.mkdir(exist_ok=True)
    print(f"Creating backup of existing files in: {backup_dir}")
    
    for filename in existing_files:
        source_file = output_path / filename
        backup_file = backup_dir / filename
        
        try:
            with open(source_file, 'r') as f:
                data = json.load(f)
            with open(backup_file, 'w') as f:
                json.dump(data, f, indent=2)
            backup_created = True
            print(f"  ✓ Backed up {filename}")
        except Exception as e:
            print(f"Warning: Could not backup {filename}: {e}")
elif backup_needed and not existing_files:
    print("No existing database files found to backup")
elif not backup_needed and existing_files:
    print("Backup disabled - existing files will be overwritten")
else:
    print("No backup needed - creating new database")

# Save compounds database
print(f"\n Saving compounds database...")
save_compounds_database(compounds, compound_index)

In [ ]:
# Create metadata file
metadata = {
    "database_info": {
        "last_updated": timestamp,
        "updated_by": creator_name,
        "notes": creation_notes,
        "operation": "update" if existing_files else "create"
    },
    "database_totals": {
        "total_compounds": len(compounds),
        "unique_inchi_keys": len(compound_index['compound_by_inchi_key'])
    },
    "database_files": {
        "compounds": str(compounds_file),
        "compound_index": str(compound_index_file),
        "metadata": str(metadata_file)
    },
    "backup_info": {
        "backup_created": backup_created,
        "backup_location": str(backup_dir) if backup_created else None,
        "backed_up_files": existing_files if backup_created else []
    }
}

with open(metadata_file, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"✓ Metadata saved to: {metadata_file}")

# Final statistics
print(f"\n Final Database Statistics:")
print(f"   Input TSV rows processed: {processing_stats.get('total_rows_processed', 0)}")
print(f"   New compounds created: {processing_stats.get('new_compounds_created', 0)}")
print(f"   Existing compounds reused: {processing_stats.get('existing_compounds_found', 0)}")
print(f"   Total compounds in database: {len(compounds)}")